# 11 — Overlap diagnostic: is the |λ₀| spike a 1/⟨L|R⟩ singularity?

**Goal.** The spectral-gap sweep in `8_loschmidt.ipynb` shows $|\lambda_0| \approx 2.2$ near $T \approx 6$,
which is unphysical (transfer-matrix eigenvalues should satisfy $|\lambda_0| \le 1$ in the normalised picture).
Two explanations are on the table:

1. **Overlap-zero singularity.** The power-method Rayleigh quotient is $\lambda_0 = \langle L|A|R\rangle / \langle L|R\rangle$.
   If the bilinear overlap $\langle L|R\rangle \to 0$ at $T^*$, the quotient diverges even though $\langle L|A|R\rangle$ stays finite.
   A Loschmidt zero is *exactly* $\langle L|R\rangle = 0$, so this would be the DQPT signal reframed correctly.

2. **Power-method convergence failure.** Near a gap closing ($|\lambda_0| \approx |\lambda_1|$) the
   PM converges slowly or stalls; $\lambda_0$ from a not-yet-converged run is not meaningful.

**Strategy.** We reuse the existing spectral-gap cache from `8_loschmidt.ipynb` (no re-running the block
method for those integer-T points). We then run `run_pm_diagnosed` — which wraps `powermethod_lr` and
returns the raw $\langle L|R\rangle$ *before* normalisation — over a fine grid around $T \in [4.5, 7.5]$.
A single check at $T = 6$ with `block_transfer_eigs` gives us `cond(S)` to confirm whether the pencil
blows up there.

**Decision output** (last cell): explicitly state whether $|\langle L|R\rangle|$ has a minimum at the same $T$
where $|\lambda_0|$ spiked and whether `cond(S)` / `niters` blow up. If yes → the spike is the overlap-zero
singularity and $T^*$ is the DQPT candidate.

## Code audit: `./` broadcasting on MPS

Task asked to confirm no MPS is normalised with `mps ./ scalar` (which broadcasts over **all** N
site tensors, scaling the norm by `scalar^N` instead of `1/scalar` — CLAUDE.md §10 trap).

**Findings:**
- `main.jl` `compute_alcaraz_entropies` (lines 206–207, 304–305): uses `psi_L ./= sqrt(norm)` —
  the broadcasting form. It is *latently wrong* but harmless **there** because `norm ≈ 1` after the
  overlap-normalised power method, so `norm^N ≈ 1`. Not touched (additive-only workflow).
- `10_benchmark_mpo.ipynb` cells 8/12/16: same `psi ./= sqrt(norm)` / `opsi ./= n_opsi` pattern,
  same latent issue, same harmless reason.
- **`block_transfer_eigs` / `lincomb_mps` (relocated here): clean** — they use `normalize(...)` and
  single-tensor `(coeff) * mps`, never `./`.
- **`run_pm_diagnosed` (this file): clean** — normalises with `(1/c) * psi_L`, a single-tensor
  scalar multiply, so the norm scales by exactly `1/c` as intended.

Conclusion: no `./`-broadcast bug affects the quantities computed in this notebook. The two
`./=` sites in `main.jl`/`nb10` are pre-existing and only safe because `norm ≈ 1`; flag for a
future cleanup but out of scope here.

In [2]:
using ITensors, ITensorMPS, ITransverse, Revise
using JLD2, Plots, LinearAlgebra, Printf
using ProgressMeter
ProgressMeter.ijulia_behavior(:clear)

includet("main.jl")
includet("dqpt_diagnostics.jl")

# shared parameters — match the 8_loschmidt spectral-gap run exactly
p      = 0.1
lambda = 1.0
dt     = 0.1
maxdim = 256
println("Dependencies loaded.")

Dependencies loaded.


## Load existing spectral-gap cache

The block-method sweep in `8_loschmidt.ipynb` already ran at $T = 1, 2, \ldots, 7$ (and T=8 was still running).
We load that cache directly — no recomputation needed for those points.

In [3]:
gapfile = "spectral_gap_alcaraz_p$(p).jld2"

if isfile(gapfile)
    gap_done = load(gapfile, "done")
    Ts_gap   = sort(collect(keys(gap_done)))
    println("Loaded $(length(Ts_gap)) cached T points: ", Ts_gap)
    println("\nT       |λ0|        |λ1|       gap_ratio   conv")
    println("-"^60)
    for T in Ts_gap
        r = gap_done[T]
        @printf("  %4.1f   %.5f    %.5f    %.4f      %s\n",
            T, r.abs[1], r.abs[2], r.gap, r.conv)
    end
else
    println("Cache file not found: $gapfile")
    println("Run the spectral-gap sweep in 8_loschmidt.ipynb first, or set gap_done = Dict().")
    gap_done = Dict{Float64, Any}()
    Ts_gap   = Float64[]
end

Loaded 8 cached T points: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]

T       |λ0|        |λ1|       gap_ratio   conv
------------------------------------------------------------
   1.0   0.89443    0.79968    0.8941      converged
   2.0   0.89418    0.88949    0.9948      converged
   3.0   0.89577    0.89296    0.9969      converged
   4.0   0.89722    0.89206    0.9943      converged
   5.0   0.89727    0.89483    0.9973      maxiter
   6.0   2.22803    0.89561    0.4020      maxiter
   7.0   1.26855    0.93947    0.7406      maxiter
   8.0   1.01060    0.89213    0.8828      maxiter


## Minimal check at T = 6 (single point)

Run `run_pm_diagnosed` and `block_transfer_eigs` at $T = 6.0$ only.
This is fast (one PM run + one block run) and tells us immediately:
- Is $|\langle L|R\rangle|$ small at this T?
- Is `cond(S)` large (ill-conditioned pencil = near-degenerate gap)?
- Does the PM declare `stuck`?

In [19]:
T_check = 6.0
println("========== Single-T check at T = $T_check ==========")

# --- powermethod_lr path ---
pm_res = run_pm_diagnosed(T_check; p=p, lambda=lambda, dt=dt, maxdim=maxdim)

println("\n--- run_pm_diagnosed ---")
println("  lr_overlap_raw  = ", pm_res.lr_overlap_raw,
        "   |·| = ", abs(pm_res.lr_overlap_raw))
println("  lr_overlap      = ", pm_res.lr_overlap,
        "   (should be ≈ 1 after normalization)")
println("  lambda0 (Rayleigh) = ", pm_res.lambda0)
println("  |lambda0|          = ", abs(pm_res.lambda0))
println("  niters  = ", pm_res.niters)
println("  stuck   = ", pm_res.stuck)
println("  final ΔS = ", isempty(pm_res.ds_hist) ? "N/A" : last(pm_res.ds_hist))
println("  max χ    = ", isempty(pm_res.chi_hist) ? "N/A" : maximum(pm_res.chi_hist))

In [20]:
# --- block_transfer_eigs at the same T. Build a FRESH (mpo, scaffold) pair here:
#     the block method seeds its own random vectors, so it only needs a mutually
#     consistent mpo+scaffold (same Index objects) — not pm_res's converged vectors.
#     Building both from one call avoids any index mismatch AND any stale pm_res.
mpo_chk, scaffold_chk = build_alcaraz_tmpo(T_check; p=p, lambda=lambda, dt=dt)

theta_blk, Lblk, Rblk, info_blk = block_transfer_eigs(mpo_chk, scaffold_chk;
    k=4, maxdim=maxdim, cutoff=1e-12, itermax=300, eps_conv=1e-8, n_track=2)

println("\n--- block_transfer_eigs ---")
println("  theta (eigen-based): ", round.(theta_blk, digits=5))
println("  |theta|:             ", round.(abs.(theta_blk), digits=5))
println("  gap_ratio = |θ₁|/|θ₀| = ", round(abs(theta_blk[2])/abs(theta_blk[1]), digits=5))
println("  cond(S) final   = ", round(info_blk[:condS], sigdigits=4))
println("  niters          = ", info_blk[:niters])
println("  reason          = ", info_blk[:reason])

# Rayleigh check: expval_LR(L,mpo,R)/overlap_lr(L,R) should match theta
rq0 = expval_LR(Lblk[1], mpo_chk, Rblk[1]) / overlap_lr(Lblk[1], Rblk[1])
println("  Rayleigh check: rq0 = ", round(rq0, digits=5),
        "  (should ≈ theta[1] = ", round(theta_blk[1], digits=5), ")")

In [23]:
# Save all diagnostic scalars to JLD2 so results survive a kernel restart.
# ITensor objects (MPS/MPO) are NOT saved — only the numbers we care about.
checkpoint_file = "checkpoint_T6_p$(p).jld2"

# gap cache is already on disk in gapfile; save a summary dict here too
gap_summary = Dict(T => (abs=gap_done[T].abs, arg=gap_done[T].arg,
                          gap=gap_done[T].gap, conv=gap_done[T].conv)
                   for T in keys(gap_done))

# pm_res scalars
pm_scalars = if @isdefined(pm_res)
    Dict(:T               => T_check,
         :lr_overlap_raw  => pm_res.lr_overlap_raw,
         :lr_overlap      => pm_res.lr_overlap,
         :lambda0         => pm_res.lambda0,
         :niters          => pm_res.niters,
         :stuck           => pm_res.stuck,
         :ds_hist         => pm_res.ds_hist,
         :chi_hist        => pm_res.chi_hist)
else
    Dict(:T => T_check, :error => "pm_res not defined")
end

# block_transfer_eigs scalars (may not exist yet)
blk_scalars = if @isdefined(theta_blk)
    Dict(:theta     => theta_blk,
         :condS     => info_blk[:condS],
         :condS_hist=> info_blk[:condS_hist],
         :niters    => info_blk[:niters],
         :reason    => info_blk[:reason],
         :dtheta    => info_blk[:dtheta])
else
    Dict(:error => "block not run yet")
end

jldsave(checkpoint_file;
        gap_summary  = gap_summary,
        pm_scalars   = pm_scalars,
        blk_scalars  = blk_scalars,
        p=p, lambda=lambda, dt=dt, maxdim=maxdim)

println("Saved to $checkpoint_file")
println("  gap_summary: $(length(gap_summary)) T points")
println("  pm_scalars keys: ", collect(keys(pm_scalars)))
println("  blk_scalars keys: ", collect(keys(blk_scalars)))

In [4]:
# ── LOAD AND PRINT after kernel restart ──────────────────────────────────────
# Run this cell (needs only JLD2 + Printf, no ITensors/ITransverse):
using JLD2, Printf

checkpoint_file = "checkpoint_T6_p0.1.jld2"
ck = load(checkpoint_file)
pm = ck["pm_scalars"]
bk = ck["blk_scalars"]
T_ck = pm[:T]

println("=== Spectral gap cache (from 8_loschmidt.ipynb) ===")
gs = ck["gap_summary"]
for T in sort(collect(keys(gs)))
    r = gs[T]
    @printf("  T=%4.1f  |λ0|=%.5f  |λ1|=%.5f  gap=%.4f  %s\n",
            T, r.abs[1], r.abs[2], r.gap, r.conv)
end

println("\n=== run_pm_diagnosed at T=$T_ck ===")
if haskey(pm, :error)
    println("  ERROR: ", pm[:error])
else
    @printf("  lr_overlap_raw   = %s\n",   pm[:lr_overlap_raw])
    @printf("  |lr_overlap_raw| = %.6f\n", abs(pm[:lr_overlap_raw]))
    @printf("  lr_overlap       = %s  (≈1 after norm)\n", pm[:lr_overlap])
    @printf("  lambda0 (Rayleigh) = %s\n", pm[:lambda0])
    @printf("  |lambda0|          = %.6f\n", abs(pm[:lambda0]))
    @printf("  niters   = %d\n", pm[:niters])
    @printf("  stuck    = %s\n", pm[:stuck])
    @printf("  final ΔS = %.3e\n", isempty(pm[:ds_hist]) ? NaN : last(pm[:ds_hist]))
    @printf("  max χ    = %d\n",   isempty(pm[:chi_hist]) ? 0   : maximum(pm[:chi_hist]))
end

println("\n=== block_transfer_eigs at T=$T_ck ===")
if haskey(bk, :error)
    println("  Not run yet: ", bk[:error])
else
    @printf("  theta:     %s\n", round.(bk[:theta], digits=5))
    @printf("  |theta|:   %s\n", round.(abs.(bk[:theta]), digits=5))
    @printf("  gap_ratio  = %.5f\n", abs(bk[:theta][2])/abs(bk[:theta][1]))
    @printf("  cond(S)    = %.3e\n", bk[:condS])
    @printf("  niters     = %d\n",   bk[:niters])
    @printf("  reason     = %s\n",   bk[:reason])
end

=== Spectral gap cache (from 8_loschmidt.ipynb) ===
  T= 1.0  |λ0|=0.89443  |λ1|=0.79968  gap=0.8941  converged
  T= 2.0  |λ0|=0.89418  |λ1|=0.88949  gap=0.9948  converged
  T= 3.0  |λ0|=0.89577  |λ1|=0.89296  gap=0.9969  converged
  T= 4.0  |λ0|=0.89722  |λ1|=0.89206  gap=0.9943  converged
  T= 5.0  |λ0|=0.89727  |λ1|=0.89483  gap=0.9973  maxiter
  T= 6.0  |λ0|=2.22803  |λ1|=0.89561  gap=0.4020  maxiter
  T= 7.0  |λ0|=1.26855  |λ1|=0.93947  gap=0.7406  maxiter
  T= 8.0  |λ0|=1.01060  |λ1|=0.89213  gap=0.8828  maxiter

=== run_pm_diagnosed at T=6.0 ===
  lr_overlap_raw   = 0.9999999999999974 - 1.339206523454095e-15im
  |lr_overlap_raw| = 1.000000
  lr_overlap       = 1.0000000000000002 + 2.2898349882893854e-16im  (≈1 after norm)
  lambda0 (Rayleigh) = -0.008393498154741394 - 0.756453206620071im
  |lambda0|          = 0.756500
  niters   = 803
  stuck    = true
  final ΔS = 2.043e-02
  max χ    = 37

=== block_transfer_eigs at T=6.0 ===
  theta:     ComplexF64[0.14879 + 1.00112im, 0.470

the power method isn't converging for T>5. what we can now do:
1. check if there is a Fisher zero (indicative of a DQPT)
2. divide the sector in two so there is a real gap and the power method can converge correctly to the most dominant eigenvalue (sustained Z2 near-degeneracy (two dominant eigenvalues equal in modulus) leaves no gap for the single-vector method to converge into)

## Full overlap sweep

Run `run_pm_diagnosed` over:
- Coarse grid $T = 1, 2, \ldots, 4$ and $T = 8, 9, 10$ (away from the feature)
- Fine grid $T = 4.5, 4.6, \ldots, 7.5$ (dt=0.1 resolution around the suspected DQPT)

Uses `crashsafe_sweep` — checkpoints after each T, re-run safe.
**Only `run_pm_diagnosed` is used here** (cheaper than block method; sufficient to track $\langle L|R\rangle$).

In [5]:
Ts_coarse = vcat(collect(1.0:1.0:4.0), collect(8.0:1.0:10.0))
Ts_fine   = collect(4.5:0.1:7.5)
Ts_sweep  = sort(unique(vcat(Ts_coarse, Ts_fine)))
println("Sweep: $(length(Ts_sweep)) T values from $(Ts_sweep[1]) to $(Ts_sweep[end])")

ovfile = "overlap_diag_p$(p)_dt$(dt).jld2"

ov_done = crashsafe_sweep(Ts_sweep; cachefile=ovfile) do T
    r = run_pm_diagnosed(T; p=p, lambda=lambda, dt=dt, maxdim=maxdim)
    nt = (lr_overlap_raw = r.lr_overlap_raw,
          lr_overlap     = r.lr_overlap,
          lambda0        = r.lambda0,
          niters         = r.niters,
          stuck          = r.stuck,
          final_ds       = isempty(r.ds_hist) ? NaN : last(r.ds_hist),
          max_chi        = isempty(r.chi_hist) ? 0   : maximum(r.chi_hist))
    println("  T=$T  |⟨L|R⟩|=$(round(abs(nt.lr_overlap_raw),digits=5))" *
            "  |λ0|=$(round(abs(nt.lambda0),digits=5))" *
            "  niters=$(nt.niters) stuck=$(nt.stuck)")
    nt
end

println("\nSweep complete: $(length(ov_done)) points cached in $ovfile")

[PM LR|RTM] L=100, cutoff=1.0e-14, maxdim=256, normalize=overlap)  24%  ETA: 1:57:26 ( 1.84  s/it)
   Info: [1181]  chi=25 | ds=0.022450973996423873 | <R|Rprev> = NaN┌ Warning: PM Stuck after 201/1182 steps | ds=0.017957104990202033 | chi=25)
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/power_method/pm.jl:151


  T=10.0  |⟨L|R⟩|=1.0  |λ0|=0.97325  niters=1182 stuck=true

Sweep complete: 38 points cached in overlap_diag_p0.1_dt0.1.jld2


In [7]:
# ── Print overlap sweep results from ov_done (already in memory) ──────────────
println("T        |⟨L|R⟩|       arg⟨L|R⟩   |λ0|Rayleigh   niters  stuck")
println("-"^75)
for T in sort(collect(keys(ov_done)))
    r = ov_done[T]
    haskey(r, :error) && (@printf("  T=%5.2f  ERROR: %s\n", T, r[:error]); continue)
    @printf("  T=%5.2f   %.5e   %+.4f      %.5f        %4d    %s\n",
            T, abs(r.lr_overlap_raw), angle(r.lr_overlap_raw),
            abs(r.lambda0), r.niters, r.stuck)
end

# also save to disk right now so results survive
jldsave("overlap_sweep_p$(p)_dt$(dt).jld2"; ov_done=ov_done, p=p, lambda=lambda, dt=dt)
println("\nSaved to overlap_sweep_p$(p)_dt$(dt).jld2")

T        |⟨L|R⟩|       arg⟨L|R⟩   |λ0|Rayleigh   niters  stuck
---------------------------------------------------------------------------
  T= 1.00   1.00000e+00   -0.0000      0.89443          56    false
  T= 2.00   1.00000e+00   -0.0000      0.89416        1004    false
  T= 3.00   1.00000e+00   +0.0000      0.89580        1666    false
  T= 4.00   1.00000e+00   -0.0000      0.89722        1194    false
  T= 4.50   1.00000e+00   +0.0000      0.89734        1087    false
  T= 4.60   1.00000e+00   -0.0000      0.89738        1053    false
  T= 4.70   1.00000e+00   -0.0000      0.89736         807    true
  T= 4.80   1.00000e+00   -0.0000      0.89735        1152    false
  T= 4.90   1.00000e+00   -0.0000      0.89732        1249    false
  T= 5.00   1.00000e+00   +0.0000      0.89736        1030    true
  T= 5.10   1.00000e+00   -0.0000      0.89729        1499    true
  T= 5.20   1.00000e+00   -0.0000      0.89734        1528    true
  T= 5.30   1.00000e+00   +0.0000      0.89735   

## condS at the spike — block method at selected T values

Run `block_transfer_eigs` (k=4) at the T values where $|\langle L|R\rangle|$ is smallest,
to get `cond(S)` and confirm the pencil singularity.

In [ ]:
# Identify T values with the smallest |⟨L|R⟩| from the sweep
Ts_ov = sort(collect(keys(ov_done)))
ov_abs = [abs(ov_done[T].lr_overlap_raw) for T in Ts_ov]

# Run block method at the 3 points with smallest overlap + T=6.0 always
perm_small = sortperm(ov_abs)[1:min(3, length(Ts_ov))]
Ts_blk = sort(unique(vcat([Ts_ov[i] for i in perm_small], [6.0])))
println("Running block_transfer_eigs at T = ", Ts_blk)

condS_file = "condS_check_p$(p).jld2"
condS_done = isfile(condS_file) ? load(condS_file, "done") : Dict{Float64, Any}()

for T in Ts_blk
    haskey(condS_done, T) && (println("  T=$T (cached)"); continue)
    mpo_blk, scaffold_blk = build_alcaraz_tmpo(T; p=p, lambda=lambda, dt=dt)
    theta_b, _, _, info_b = block_transfer_eigs(mpo_blk, scaffold_blk;
        k=4, maxdim=maxdim, cutoff=1e-12, itermax=300, eps_conv=1e-8, n_track=2)
    condS_done[T] = (abs=abs.(theta_b), gap=abs(theta_b[2])/abs(theta_b[1]),
                     condS=info_b[:condS], niters=info_b[:niters], reason=info_b[:reason])
    jldsave(condS_file; done=condS_done)
    r = condS_done[T]
    println("  T=$T  |θ0|=$(round(r.abs[1],digits=5))  gap=$(round(r.gap,digits=4))" *
            "  cond(S)=$(round(r.condS,sigdigits=3))  niters=$(r.niters)  $(r.reason)")
    GC.gc()
end

Running block_transfer_eigs at T = 

In [ ]:
# ── Print overlap sweep results from ov_done (already in memory) ──────────────
println("T        |⟨L|R⟩|       arg⟨L|R⟩   |λ0|Rayleigh   niters  stuck")
println("-"^75)
for T in sort(collect(keys(ov_done)))
    r = ov_done[T]
    haskey(r, :error) && (@printf("  T=%5.2f  ERROR: %s\n", T, r[:error]); continue)
    @printf("  T=%5.2f   %.5e   %+.4f      %.5f        %4d    %s\n",
            T, abs(r.lr_overlap_raw), angle(r.lr_overlap_raw),
            abs(r.lambda0), r.niters, r.stuck)
end

# also save to disk right now so results survive
jldsave("overlap_sweep_p$(p)_dt$(dt).jld2"; ov_done=ov_done, p=p, lambda=lambda, dt=dt)
println("\nSaved to overlap_sweep_p$(p)_dt$(dt).jld2")

## 4-panel diagnostic figure

In [ ]:
# Assemble plotting arrays from the overlap sweep
Ts_plt     = sort(collect(keys(ov_done)))
ov_raw_abs = [abs(ov_done[T].lr_overlap_raw) for T in Ts_plt]
ov_raw_arg = [angle(ov_done[T].lr_overlap_raw) for T in Ts_plt]
lam0_ray   = [abs(ov_done[T].lambda0)  for T in Ts_plt]   # Rayleigh |λ0|
lam0_niters= [ov_done[T].niters        for T in Ts_plt]
lam0_stuck = [ov_done[T].stuck         for T in Ts_plt]

# block-method eigen-based |λ0| at the checked points
Ts_blk_plt  = sort(collect(keys(condS_done)))
lam0_eigen  = [condS_done[T].abs[1] for T in Ts_blk_plt]
condS_vals  = [condS_done[T].condS  for T in Ts_blk_plt]

# ── panel 1: |⟨L|R⟩| vs T
p1 = plot(Ts_plt, ov_raw_abs, lw=2, marker=:circle, color=:blue, label="|⟨L|R⟩|",
          xlabel="T", ylabel="|⟨L|R⟩|", title="Biorthogonal overlap",
          yscale=:log10, grid=true, framestyle=:box, legend=:topright)
vline!(p1, [6.0], ls=:dot, color=:red, alpha=0.6, label="T=6")

# ── panel 2: arg⟨L|R⟩ vs T
p2 = plot(Ts_plt, ov_raw_arg, lw=2, marker=:circle, color=:darkorange, label="arg⟨L|R⟩",
          xlabel="T", ylabel="arg⟨L|R⟩ (rad)", title="Phase of overlap",
          grid=true, framestyle=:box, legend=:topright)
hline!(p2, [π, -π], ls=:dash, color=:gray, alpha=0.4, label="±π")
vline!(p2, [6.0], ls=:dot, color=:red, alpha=0.6, label="T=6")

# ── panel 3: cond(S) and niters vs T (two y-axes)
p3 = plot(Ts_plt, lam0_niters, lw=2, marker=:utriangle, color=:purple, label="niters (PM)",
          xlabel="T", ylabel="niters", title="PM iterations & cond(S)",
          grid=true, framestyle=:box, legend=:topleft)
if !isempty(Ts_blk_plt)
    plot!(twinx(p3), Ts_blk_plt, log10.(condS_vals), lw=2, marker=:diamond,
          color=:green, label="log₁₀ cond(S)", ylabel="log₁₀ cond(S)", legend=:topright)
end
vline!(p3, [6.0], ls=:dot, color=:red, alpha=0.6, label="")
# mark stuck points
stuck_Ts = [Ts_plt[i] for i in eachindex(Ts_plt) if lam0_stuck[i]]
isempty(stuck_Ts) || scatter!(p3, stuck_Ts, fill(0, length(stuck_Ts)),
    marker=:x, color=:red, ms=8, label="PM stuck", legend=:topleft)

# ── panel 4: Rayleigh |λ0| vs eigen-based |λ0|
p4 = plot(Ts_plt, lam0_ray, lw=2, marker=:circle, color=:black,
          label="|λ0| Rayleigh",
          xlabel="T", ylabel="|λ0|", title="Rayleigh vs eigen-based |λ0|",
          grid=true, framestyle=:box, legend=:topright)
if !isempty(Ts_blk_plt)
    scatter!(p4, Ts_blk_plt, lam0_eigen, marker=:star5, ms=8,
             color=:red, label="|λ0| eigen (block)")
end
hline!(p4, [1.0], ls=:dash, color=:gray, alpha=0.5, label="|λ|=1")
vline!(p4, [6.0], ls=:dot, color=:red, alpha=0.6, label="")

fig = plot_panels(p1, p2, p3, p4;
    filename="overlap_diag_p$(p)_dt$(dt).png",
    title="Overlap diagnostic — Alcaraz p=$p, dt=$dt",
    fig_size=(1900, 480))
display(fig)

## Decision output

In [ ]:
# ── find T where |⟨L|R⟩| is minimum
imin     = argmin(ov_raw_abs)
T_min_ov = Ts_plt[imin]
val_min  = ov_raw_abs[imin]

# ── find T where Rayleigh |λ0| is maximum (the spike)
imax_ray  = argmax(lam0_ray)
T_max_ray = Ts_plt[imax_ray]
val_max_ray = lam0_ray[imax_ray]

# ── condS at T_min_ov (if we ran block there)
condS_at_min = haskey(condS_done, T_min_ov) ? condS_done[T_min_ov].condS : NaN
condS_at_6   = haskey(condS_done, 6.0)      ? condS_done[6.0].condS      : NaN

# ── stuck at T_min_ov
stuck_at_min = haskey(ov_done, T_min_ov) ? ov_done[T_min_ov].stuck : missing

println("="^65)
println("DECISION OUTPUT — overlap diagnostic (p=$p, dt=$dt)")
println("="^65)
@printf("  |⟨L|R⟩| minimum at T = %.2f  (value = %.3e)\n", T_min_ov, val_min)
@printf("  Rayleigh |λ0| spike  at T = %.2f  (value = %.3f)\n", T_max_ray, val_max_ray)
@printf("  cond(S) at T_min = %.2e   at T=6: %.2e\n", condS_at_min, condS_at_6)
@printf("  PM stuck at T_min_ov? %s\n", string(stuck_at_min))
println()

same_T  = abs(T_min_ov - T_max_ray) < 0.55   # within one coarse step
condS_large = isfinite(condS_at_6) && condS_at_6 > 1e6

if same_T && condS_large
    println("CONCLUSION: the |λ0| spike IS the overlap-zero singularity.")
    println("  |⟨L|R⟩| → 0 and cond(S) blows up at the same T* ≈ $T_min_ov.")
    println("  This T* is the DQPT candidate: the Loschmidt zero in the transverse picture.")
    println("  Proceed to 12_dt_convergence to check stability as dt → 0.")
elseif same_T && !condS_large
    println("CONCLUSION: |⟨L|R⟩| dips at T* ≈ $T_min_ov but cond(S) is moderate.")
    println("  Likely a genuine overlap zero (DQPT), not a pure convergence failure.")
    println("  Run 12_dt_convergence for confirmation.")
elseif !same_T && condS_large
    println("CONCLUSION: spike at T=$(T_max_ray) and overlap minimum at T=$(T_min_ov) disagree.")
    println("  cond(S) is large → likely a convergence failure, not a clean DQPT.")
    println("  Investigate with larger maxdim or longer PM run before proceeding.")
else
    println("CONCLUSION: inconclusive — manual inspection of the panels required.")
    println("  T_spike=$(T_max_ray), T_min_overlap=$(T_min_ov), condS_at_6=$(condS_at_6).")
end
println("="^65)